# Sequential Binary Experiments

This notebook is the original staged binary classification workflow for the protease cleavage site project.

It records the first complete sequence of experiments:

1. compare cleavage site clustering options;
2. compare negative pair construction strategies;
3. compare frozen and unfrozen ESM2 8M architectures;
4. run heldout evaluation only after validation-based selection.

Use this notebook as the historical/completed-results workflow. For the final follow-up workflow with MEROPS family parsing, pair manifests, reproducibility fingerprints, validation sensitivity checks, and optional 150M runs, use `sequential_binary_followup_experiments.ipynb`.

## 1. Colab Drive Mount

Mount Google Drive when running in Colab. This is where the large experiment outputs were saved during GPU runs.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 2. Imports And Run Settings

Set the active stage, model, seeds, learning rates, batch size, and output paths before launching a run.

In [2]:
from pathlib import Path
from contextlib import nullcontext
from dataclasses import dataclass
import gc
import json
import math
import random
import re

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
from transformers import AutoModel, AutoTokenizer
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    matthews_corrcoef,
    confusion_matrix,
)

PROJECT_ROOT_OVERRIDE = None

# Main run switches. Change these before running a stage.
MODEL_NAME = 'facebook/esm2_t6_8M_UR50D'
RUN_STAGE = 'architecture'  # clustering, sampling, architecture, all
BEST_CLUSTERING_OVERRIDE = 'no_clustering'
BEST_NEGATIVE_STRATEGY_OVERRIDE = 'cross_family'
BEST_ARCHITECTURE_FREEZE_OVERRIDE = None
SMOKE_TEST = False
SEEDS = [42]
EPOCHS = 1 if SMOKE_TEST else 20
BATCH_SIZE = 4
LR_FROZEN = 1e-4
LR_UNFROZEN = 1e-7
WEIGHT_DECAY = 1e-4
MAX_LEN_PROTEASE = 1022
MAX_LEN_SITE = 16
DROPOUT = 0.20
HIDDEN_DIM = 256
NEGATIVES_PER_POSITIVE = 1
MAX_POSITIVE_ROWS = 512 if SMOKE_TEST else None
THRESHOLDS = np.linspace(0.05, 0.95, 19)

EVALUATE_HELDOUT_EVERY_RUN = False
MIN_VALID_CODES_WARNING = 5
MIN_VALID_POSITIVES_WARNING = 100

P1_MIN_NATURAL_PER_PROTEASE = 10
P1_NONPHYSIO_CAP_PER_CLUSTER = 2000
NATURAL_TYPES = {'physiological', 'pathological'}
AMINO_ACIDS = list('ARNDCQEGHILKMFPSTWYV')
AA_TO_P1_CLUSTER = {
    'A': 1, 'V': 1, 'I': 1, 'L': 1, 'M': 1,
    'F': 2, 'Y': 2, 'W': 2,
    'S': 3, 'T': 3, 'N': 3, 'Q': 3,
    'D': 4, 'E': 4,
    'K': 5, 'R': 5,
    'H': 6,
    'G': 7,
    'P': 8,
    'C': 9,
}

def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def find_project_root() -> Path:
    if PROJECT_ROOT_OVERRIDE is not None:
        return Path(PROJECT_ROOT_OVERRIDE)
    candidates = [
        Path.cwd(),
        Path.cwd().parent,
        Path('/content/deep-learning-proteases'),
        Path('/content/drive/MyDrive/deep-learning-proteases'),
        Path('/content/drive/MyDrive/MEIN40430 Materials/deep-learning-proteases'),
    ]
    for candidate in candidates:
        if (candidate / 'processed_data' / 'training.csv').exists():
            return candidate
    search_roots = [Path.cwd(), Path('/content')]
    if Path('/content/drive/MyDrive').exists():
        search_roots.append(Path('/content/drive/MyDrive'))
    for root in search_roots:
        try:
            matches = list(root.glob('**/processed_data/training.csv'))
        except Exception:
            matches = []
        if matches:
            return matches[0].parents[1]
    raise FileNotFoundError('Could not find processed_data/training.csv.')

PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / 'processed_data'
# Large outputs are written outside Git and usually synced through Google Drive.
OUTPUT_DIR = PROJECT_ROOT / 'outputs' / 'sequential_binary_experiments'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_CSV = DATA_DIR / 'training.csv'
TEST_SINGLETON_CSV = DATA_DIR / 'test_1_sample.csv'
TEST_LOW_SAMPLE_CSV = DATA_DIR / 'test_2_to_10_samples.csv'
TRAIN_8MER_CSV = DATA_DIR / 'training_8mer_clustering.csv'

print(f'Project root: {PROJECT_ROOT}')
print(f'Output dir:   {OUTPUT_DIR}')
print(f'Run stage:    {RUN_STAGE}')
print(f'Smoke test:   {SMOKE_TEST}')

Project root: /content/drive/MyDrive/MEIN40430 Materials/deep-learning-proteases
Output dir:   /content/drive/MyDrive/MEIN40430 Materials/deep-learning-proteases/outputs/sequential_binary_experiments
Run stage:    architecture
Smoke test:   False


## 3. Data Helpers

Clean protease codes and cleavage-site strings, load MEROPS-derived CSVs, and identify held-out protease codes.

In [3]:
def normalize_code(value) -> str:
    return str(value).strip()

def normalize_site(value) -> str:
    return str(value).strip().upper().replace(' ', '')

def normalize_pair(code, site):
    return (normalize_code(code), normalize_site(site))

def clean_site(site: str) -> str:
    return normalize_site(site)

def clean_for_esm(seq: str) -> str:
    seq = str(seq).upper().replace(' ', '')
    return seq.replace('-', 'X')

def load_processed_csv(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path)
    required = {'code', 'site', 'protease', 'cleavage_type'}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f'{path.name} is missing columns: {sorted(missing)}')
    df = df.dropna(subset=['code', 'site', 'protease']).copy()
    df['code'] = df['code'].map(normalize_code)
    df['site'] = df['site'].map(normalize_site)
    df['protease'] = df['protease'].astype(str).str.strip()
    df['cleavage_type'] = df['cleavage_type'].astype(str).str.strip()
    df['family'] = df['code'].str[0]
    return df

def find_p1_csv() -> Path | None:
    candidates = [
        DATA_DIR / 'training_p1_clustering.csv',
        DATA_DIR / 'training_p1_clustered.csv',
        DATA_DIR / 'training_p1_site_clustering.csv',
        DATA_DIR / 'training_1mer_clustering.csv',
    ]
    for path in candidates:
        if path.exists():
            return path
    matches = sorted(DATA_DIR.glob('*p1*clustering*.csv')) + sorted(DATA_DIR.glob('*1mer*clustering*.csv'))
    return matches[0] if matches else None

def p1_residue(site: str) -> str:
    site = clean_site(site)
    return site[3] if len(site) >= 4 else 'X'

def add_p1_cluster(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out['p1_aa'] = out['site'].map(p1_residue)
    out['p1_cluster'] = out['p1_aa'].map(AA_TO_P1_CLUSTER).fillna(0).astype(int)
    return out

def select_p1_positive_rows(df: pd.DataFrame, seed: int) -> pd.DataFrame:
    df = add_p1_cluster(df)
    natural_mask = df['cleavage_type'].isin(NATURAL_TYPES)
    natural_counts = df[natural_mask].groupby('code').size()
    valid_codes = natural_counts[natural_counts >= P1_MIN_NATURAL_PER_PROTEASE].index.tolist()
    natural_samples = df[df['code'].isin(valid_codes) & natural_mask].copy()
    available_nonphysio = df[df['code'].isin(valid_codes) & (df['cleavage_type'] == 'non-physiological')].copy()
    selected = [natural_samples]
    for cluster_id in sorted(c for c in df['p1_cluster'].dropna().unique() if c != 0):
        cluster_rows = available_nonphysio[available_nonphysio['p1_cluster'] == cluster_id]
        n_take = min(P1_NONPHYSIO_CAP_PER_CLUSTER, len(cluster_rows))
        if n_take > 0:
            selected.append(cluster_rows.sample(n=n_take, random_state=seed))
    out = pd.concat(selected, ignore_index=True)
    subset = ['code', 'site', 'cleavageID'] if 'cleavageID' in out.columns else ['code', 'site']
    return out.drop_duplicates(subset=subset).reset_index(drop=True)

test_singleton_raw = load_processed_csv(TEST_SINGLETON_CSV)
test_low_sample_raw = load_processed_csv(TEST_LOW_SAMPLE_CSV)
HELDOUT_CODES = set(test_singleton_raw['code']).union(set(test_low_sample_raw['code']))

P1_CSV = find_p1_csv()
print(f'P1 CSV: {P1_CSV if P1_CSV else "not found; using internal P1 selection"}')
print(f'8-mer CSV exists: {TRAIN_8MER_CSV.exists()}')
print(f'Held-out protease codes: {len(HELDOUT_CODES):,}')

P1 CSV: /content/drive/MyDrive/MEIN40430 Materials/deep-learning-proteases/processed_data/training_p1_clustering.csv
8-mer CSV exists: True
Held-out protease codes: 557


## 4. Positive Source Selection

Load the chosen positive-record source: no clustering, P1/1-mer selection, or 8-mer clustering selection.

In [4]:
positive_source_cache = {}

def source_path_for_name(source_name: str) -> Path | None:
    if source_name == 'no_clustering':
        return TRAIN_CSV
    if source_name == 'p1_clustering':
        return P1_CSV
    if source_name == '8mer_clustering':
        return TRAIN_8MER_CSV
    raise ValueError(f'Unknown source: {source_name}')

def load_positive_source(source_name: str, seed: int = 42) -> tuple[pd.DataFrame, dict]:
    cache_key = (source_name, seed)
    if cache_key in positive_source_cache:
        frame, info = positive_source_cache[cache_key]
        return frame.copy(), info.copy()

    source_path = source_path_for_name(source_name)
    used_internal_p1 = False
    if source_name == 'p1_clustering' and source_path is None:
        base = load_processed_csv(TRAIN_CSV)
        df = select_p1_positive_rows(base, seed)
        used_internal_p1 = True
    else:
        if source_path is None or not source_path.exists():
            raise FileNotFoundError(
                f'Missing {source_path}. Add the required clustering CSV to processed_data first.'
            )
        df = load_processed_csv(source_path)

    rows_before = len(df)
    codes_before = int(df['code'].nunique())
    df = df[~df['code'].isin(HELDOUT_CODES)].copy().reset_index(drop=True)
    rows_after = len(df)
    codes_after = int(df['code'].nunique())
    info = {
        'source_name': source_name,
        'source_path': str(source_path) if source_path is not None else None,
        'used_internal_p1_selection': used_internal_p1,
        'rows_before_heldout_exclusion': rows_before,
        'rows_after_heldout_exclusion': rows_after,
        'rows_removed_by_heldout_exclusion': rows_before - rows_after,
        'codes_before_heldout_exclusion': codes_before,
        'codes_after_heldout_exclusion': codes_after,
        'codes_removed_by_heldout_exclusion': codes_before - codes_after,
        'unique_sites_after_heldout_exclusion': int(df['site'].nunique()),
    }
    positive_source_cache[cache_key] = (df.copy(), info.copy())
    return df, info

def build_global_known_positive_pairs() -> set:
    paths = [TRAIN_CSV, TEST_SINGLETON_CSV, TEST_LOW_SAMPLE_CSV]
    if P1_CSV is not None:
        paths.append(P1_CSV)
    if TRAIN_8MER_CSV.exists():
        paths.append(TRAIN_8MER_CSV)
    pairs = set()
    for path in paths:
        if not path.exists():
            continue
        df = pd.read_csv(path, usecols=lambda col: col in {'code', 'site'})
        if not {'code', 'site'}.issubset(df.columns):
            continue
        for code, site in zip(df['code'], df['site']):
            pairs.add(normalize_pair(code, site))
    return pairs

GLOBAL_KNOWN_POSITIVE_PAIRS = build_global_known_positive_pairs()
print(f'Known positive pairs blocked from negatives: {len(GLOBAL_KNOWN_POSITIVE_PAIRS):,}')

# p1_clustering corresponds to devised 1-mer/P1 clustering condition.
for source_name in ['no_clustering', 'p1_clustering']:
    df_source, info = load_positive_source(source_name, seed=SEEDS[0])
    print(f'{source_name}: {len(df_source):,} rows, {df_source["code"].nunique():,} codes, {df_source["site"].nunique():,} sites')
if TRAIN_8MER_CSV.exists():
    df_source, info = load_positive_source('8mer_clustering', seed=SEEDS[0])
    print(f'8mer_clustering: {len(df_source):,} rows, {df_source["code"].nunique():,} codes, {df_source["site"].nunique():,} sites')
else:
    print('8mer_clustering: missing training_8mer_clustering.csv')

Known positive pairs blocked from negatives: 59,106
no_clustering: 61,436 rows, 355 codes, 47,220 sites
p1_clustering: 28,518 rows, 136 codes, 22,758 sites
8mer_clustering: 32,168 rows, 136 codes, 25,589 sites


## 5. Splits And Negative Pairs

Split protease codes into train/validation groups, then build binary positive/negative pair tables for each partition.

In [5]:
def random_site_like(length: int, rng: np.random.Generator) -> str:
    return ''.join(rng.choice(AMINO_ACIDS, size=max(1, length)).tolist())

def blocked_pair(code, site) -> bool:
    return normalize_pair(code, site) in GLOBAL_KNOWN_POSITIVE_PAIRS

def scramble_site_for_code(code: str, site: str, rng: np.random.Generator) -> str:
    chars = list(clean_site(site))
    if len(chars) < 2:
        candidate = random_site_like(8, rng)
        return candidate
    original = ''.join(chars)
    for _ in range(100):
        rng.shuffle(chars)
        scrambled = ''.join(chars)
        if scrambled != original and not blocked_pair(code, scrambled):
            return scrambled
    for _ in range(100):
        candidate = random_site_like(len(original), rng)
        if not blocked_pair(code, candidate):
            return candidate
    return original[::-1]

def choose_random_site(row, candidates: pd.DataFrame, rng: np.random.Generator) -> str:
    code = row['code']
    if candidates.empty:
        return scramble_site_for_code(code, row['site'], rng)
    for _ in range(200):
        candidate = candidates.iloc[int(rng.integers(0, len(candidates)))]
        site = candidate['site']
        if not blocked_pair(code, site):
            return site
    return scramble_site_for_code(code, row['site'], rng)

def choose_cross_family_site(row, candidates: pd.DataFrame, rng: np.random.Generator) -> str:
    pool = candidates[candidates['family'] != row['family']]
    if pool.empty:
        pool = candidates
    return choose_random_site(row, pool, rng)

def make_binary_frame(
    positives: pd.DataFrame,
    negative_strategy: str,
    negatives_per_positive: int,
    seed: int,
) -> pd.DataFrame:
    positives = positives.dropna(subset=['code', 'site', 'protease']).copy().reset_index(drop=True)
    positives['label'] = 1
    positives['pair_type'] = 'positive_observed'
    positives['source_positive_site'] = positives['site']
    rng = np.random.default_rng(seed)
    strategies = ['scramble', 'random_site', 'cross_family'] if negative_strategy == 'mixed' else [negative_strategy]
    negatives = []
    for idx, row in positives.iterrows():
        for neg_idx in range(negatives_per_positive):
            strategy = strategies[(idx + neg_idx) % len(strategies)]
            if strategy == 'scramble':
                negative_site = scramble_site_for_code(row['code'], row['site'], rng)
            elif strategy == 'random_site':
                negative_site = choose_random_site(row, positives, rng)
            elif strategy == 'cross_family':
                negative_site = choose_cross_family_site(row, positives, rng)
            else:
                raise ValueError(f'Unknown negative strategy: {strategy}')
            if blocked_pair(row['code'], negative_site):
                negative_site = scramble_site_for_code(row['code'], row['site'], rng)
            neg = row.copy()
            neg['site'] = clean_site(negative_site)
            neg['label'] = 0
            neg['pair_type'] = f'negative_{strategy}'
            negatives.append(neg)
    binary = pd.concat([positives, pd.DataFrame(negatives)], ignore_index=True)
    return binary.sample(frac=1, random_state=seed).reset_index(drop=True)

def dataset_size_info(prefix: str, positives: pd.DataFrame, pairs: pd.DataFrame | None = None) -> dict:
    info = {
        f'{prefix}_positives': int(len(positives)),
        f'{prefix}_unique_codes': int(positives['code'].nunique()),
        f'{prefix}_unique_sites': int(positives['site'].nunique()),
    }
    if pairs is not None:
        info[f'{prefix}_negatives'] = int((pairs['label'] == 0).sum())
        info[f'{prefix}_pairs'] = int(len(pairs))
    return info

def split_and_make_pairs(positive_rows: pd.DataFrame, negative_strategy: str, seed: int):
    if MAX_POSITIVE_ROWS is not None and len(positive_rows) > MAX_POSITIVE_ROWS:
        positive_rows = positive_rows.sample(n=MAX_POSITIVE_ROWS, random_state=seed).reset_index(drop=True)
    n_codes = positive_rows['code'].nunique()
    if n_codes < 2:
        raise ValueError('Need at least two protease codes for grouped split.')
    # Group split tests generalisation to unseen proteases.
    splitter = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=seed)
    train_idx, valid_idx = next(splitter.split(positive_rows, groups=positive_rows['code']))
    train_positive = positive_rows.iloc[train_idx].reset_index(drop=True)
    valid_positive = positive_rows.iloc[valid_idx].reset_index(drop=True)
    train_codes = set(train_positive['code'])
    valid_codes = set(valid_positive['code'])
    assert train_codes.isdisjoint(valid_codes), 'Train and validation codes overlap.'
    assert train_codes.isdisjoint(HELDOUT_CODES), 'Held-out code found in training.'
    assert valid_codes.isdisjoint(HELDOUT_CODES), 'Held-out code found in validation.'
    warnings = []
    if len(valid_codes) < MIN_VALID_CODES_WARNING:
        warnings.append(f'Validation has only {len(valid_codes)} protease codes.')
    if len(valid_positive) < MIN_VALID_POSITIVES_WARNING:
        warnings.append(f'Validation has only {len(valid_positive)} positive examples.')
    train_pairs = make_binary_frame(train_positive, negative_strategy, NEGATIVES_PER_POSITIVE, seed)
    valid_pairs = make_binary_frame(valid_positive, negative_strategy, NEGATIVES_PER_POSITIVE, seed + 1)
    size_info = {}
    size_info.update(dataset_size_info('train', train_positive, train_pairs))
    size_info.update(dataset_size_info('valid', valid_positive, valid_pairs))
    return train_positive, valid_positive, train_pairs, valid_pairs, warnings, size_info

## 6. Binary ESM2 Classifier

Encode the protease and cleavage site with ESM2, concatenate their embeddings, and train a small classifier head.

In [6]:
class BinaryProteinPairDataset(Dataset):
    def __init__(self, frame: pd.DataFrame, tokenizer, max_len_protease=1022, max_len_site=16):
        self.frame = frame.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_len_protease = max_len_protease
        self.max_len_site = max_len_site

    def __len__(self):
        return len(self.frame)

    def encode(self, seq: str, max_len: int):
        return self.tokenizer(
            clean_for_esm(seq),
            padding='max_length',
            truncation=True,
            max_length=max_len,
            return_tensors='pt',
        )

    def __getitem__(self, idx):
        row = self.frame.iloc[idx]
        protease = self.encode(row['protease'], self.max_len_protease)
        site = self.encode(row['site'], self.max_len_site)
        return {
            'protease': {k: v.squeeze(0) for k, v in protease.items()},
            'site': {k: v.squeeze(0) for k, v in site.items()},
            'label': torch.tensor(row['label'], dtype=torch.float32),
        }

class BinaryESMClassifier(nn.Module):
    def __init__(self, model_name: str, freeze_esm: bool, hidden_dim: int, dropout: float):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        self.freeze_esm = freeze_esm
        hidden_size = self.encoder.config.hidden_size
        if freeze_esm:
            for param in self.encoder.parameters():
                param.requires_grad = False
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size * 4, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1),
        )

    @staticmethod
    def mean_pool(last_hidden_state, attention_mask):
        mask = attention_mask.unsqueeze(-1).float()
        summed = (last_hidden_state * mask).sum(dim=1)
        denom = mask.sum(dim=1).clamp(min=1e-9)
        return summed / denom

    def train(self, mode: bool = True):
        super().train(mode)
        if self.freeze_esm:
            self.encoder.eval()
        return self

    def encode(self, batch):
        context = torch.no_grad() if self.freeze_esm else nullcontext()
        with context:
            outputs = self.encoder(**batch)
            pooled = self.mean_pool(outputs.last_hidden_state, batch['attention_mask'])
        return F.normalize(pooled, p=2, dim=-1)

    def forward(self, protease_batch, site_batch):
        protease_emb = self.encode(protease_batch)
        site_emb = self.encode(site_batch)
        features = torch.cat([
            protease_emb,
            site_emb,
            torch.abs(protease_emb - site_emb),
            protease_emb * site_emb,
        ], dim=-1)
        return self.classifier(features).squeeze(-1)

def get_device():
    if torch.cuda.is_available():
        return torch.device('cuda')
    if hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
        return torch.device('mps')
    return torch.device('cpu')

def move_inputs(inputs, device):
    return {k: v.to(device) for k, v in inputs.items()}

def count_parameters(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return {'total': int(total), 'trainable': int(trainable), 'frozen': int(total - trainable)}

## 7. Metrics And Threshold Selection

Evaluate candidate thresholds and select checkpoints using validation MCC. Other metrics are tracked for interpretation.

In [7]:
def metrics_at_threshold(labels, probs, threshold):
    labels = np.asarray(labels).astype(int)
    probs = np.asarray(probs).astype(float)
    preds = (probs >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(labels, preds, labels=[0, 1]).ravel()
    row = {
        'threshold': float(threshold),
        'mcc': float(matthews_corrcoef(labels, preds)),
        'precision': float(precision_score(labels, preds, zero_division=0)),
        'recall': float(recall_score(labels, preds, zero_division=0)),
        'f1': float(f1_score(labels, preds, zero_division=0)),
        'accuracy': float(accuracy_score(labels, preds)),
        'tn': int(tn),
        'fp': int(fp),
        'fn': int(fn),
        'tp': int(tp),
    }
    try:
        row['roc_auc'] = float(roc_auc_score(labels, probs))
    except ValueError:
        row['roc_auc'] = float('nan')
    return row

def select_validation_metrics(labels, probs):
    rows = [metrics_at_threshold(labels, probs, threshold) for threshold in THRESHOLDS]
    return max(rows, key=lambda row: row['mcc'])

def metrics_with_fixed_threshold(labels, probs, threshold):
    return metrics_at_threshold(labels, probs, threshold)

def add_loss(metrics: dict, losses: list) -> dict:
    metrics = metrics.copy()
    metrics['loss'] = float(np.mean(losses)) if losses else float('nan')
    return metrics

@torch.no_grad()
def predict_model(model, loader, device, loss_fn=None):
    model.eval()
    all_probs = []
    all_labels = []
    losses = []
    for batch in tqdm(loader, desc='eval', leave=False):
        protease = move_inputs(batch['protease'], device)
        site = move_inputs(batch['site'], device)
        labels = batch['label'].to(device)
        logits = model(protease, site)
        if loss_fn is not None:
            losses.append(loss_fn(logits, labels).item())
        probs = torch.sigmoid(logits).detach().cpu().numpy()
        all_probs.extend(probs.tolist())
        all_labels.extend(labels.detach().cpu().numpy().tolist())
    return np.asarray(all_labels), np.asarray(all_probs), losses

def train_one_epoch(model, loader, optimizer, device, loss_fn):
    model.train()
    running = []
    pbar = tqdm(loader, desc='train')
    for batch in pbar:
        protease = move_inputs(batch['protease'], device)
        site = move_inputs(batch['site'], device)
        labels = batch['label'].to(device)
        optimizer.zero_grad(set_to_none=True)
        logits = model(protease, site)
        loss = loss_fn(logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        running.append(loss.item())
        pbar.set_postfix(loss=f'{np.mean(running):.4f}')
    return float(np.mean(running))

def prediction_frame(frame: pd.DataFrame, probs, threshold: float) -> pd.DataFrame:
    out = frame[['code', 'family', 'site', 'source_positive_site', 'pair_type', 'label']].copy()
    out['probability'] = probs
    out['threshold'] = threshold
    out['prediction'] = (out['probability'] >= threshold).astype(int)
    return out

def metric_value(value):
    if isinstance(value, float):
        if math.isnan(value):
            return 'nan'
        return f'{value:.4f}'
    return str(value)

def markdown_table(rows, columns):
    header = '| ' + ' | '.join(columns) + ' |'
    divider = '| ' + ' | '.join(['---'] * len(columns)) + ' |'
    body = []
    for row in rows:
        body.append('| ' + ' | '.join(metric_value(row.get(col, '')) for col in columns) + ' |')
    return '\n'.join([header, divider] + body)

## 8. Run Helpers And Summaries

Create run IDs, save metrics, record checkpoints, and keep each experiment result traceable.

In [8]:
@dataclass(frozen=True)
class ExperimentConfig:
    stage: str
    clustering: str
    negative_strategy: str
    freeze_esm: bool
    seed: int

def architecture_name(freeze_esm: bool) -> str:
    return 'frozen_esm' if freeze_esm else 'unfrozen_esm'

def safe_run_id(config: ExperimentConfig) -> str:
    text = f'{config.clustering}__{config.negative_strategy}__{architecture_name(config.freeze_esm)}__seed{config.seed}'
    return re.sub(r'[^A-Za-z0-9_.-]+', '_', text)

def make_json_safe(value):
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, dict):
        return {str(k): make_json_safe(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [make_json_safe(v) for v in value]
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        return float(value)
    return value

def save_run_summary(run_dir: Path, summary: dict, history_df: pd.DataFrame, heldout_df: pd.DataFrame | None):
    json_path = run_dir / 'run_summary.json'
    md_path = run_dir / 'run_summary.md'
    json_path.write_text(json.dumps(make_json_safe(summary), indent=2), encoding='utf-8')
    metric_columns = ['loss', 'threshold', 'mcc', 'precision', 'recall', 'f1', 'accuracy', 'roc_auc', 'tn', 'fp', 'fn', 'tp']
    lines = [
        '# Run Summary',
        '',
        '## Config',
        f'- Stage: `{summary["stage"]}`',
        f'- Clustering: `{summary["clustering"]}`',
        f'- Negative strategy: `{summary["negative_strategy"]}`',
        f'- Architecture: `{summary["architecture"]}`',
        f'- Seed: `{summary["seed"]}`',
        f'- Selection metric: `validation_mcc`',
        '',
        '## Validation',
        markdown_table([summary['validation_metrics']], metric_columns),
        '',
        '## Dataset Sizes',
        markdown_table([summary['dataset_size']], sorted(summary['dataset_size'].keys())),
        '',
    ]
    if summary.get('warnings'):
        lines.extend(['## Warnings'] + [f'- {warning}' for warning in summary['warnings']] + [''])
    if heldout_df is not None and not heldout_df.empty:
        lines.extend(['## Held-out', markdown_table(heldout_df.to_dict(orient='records'), ['dataset', 'used_for_selection'] + metric_columns), ''])
    md_path.write_text('\n'.join(lines) + '\n', encoding='utf-8')

def build_loaders(train_pairs, valid_pairs, tokenizer):
    train_dataset = BinaryProteinPairDataset(train_pairs, tokenizer, MAX_LEN_PROTEASE, MAX_LEN_SITE)
    valid_dataset = BinaryProteinPairDataset(valid_pairs, tokenizer, MAX_LEN_PROTEASE, MAX_LEN_SITE)
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    valid_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE, shuffle=False)
    return train_loader, valid_loader

def evaluate_pairs(model, frame: pd.DataFrame, tokenizer, device, loss_fn, threshold: float):
    dataset = BinaryProteinPairDataset(frame, tokenizer, MAX_LEN_PROTEASE, MAX_LEN_SITE)
    loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False)
    labels, probs, losses = predict_model(model, loader, device, loss_fn)
    metrics = add_loss(metrics_with_fixed_threshold(labels, probs, threshold), losses)
    preds = prediction_frame(frame, probs, threshold)
    return metrics, preds

def make_heldout_pairs(positive_frame: pd.DataFrame, negative_strategy: str, seed: int) -> pd.DataFrame:
    return make_binary_frame(positive_frame, negative_strategy, NEGATIVES_PER_POSITIVE, seed)

def train_experiment(config: ExperimentConfig, tokenizer, evaluate_heldout: bool = False) -> dict:
    set_seed(config.seed)
    run_id = safe_run_id(config)
    run_dir = OUTPUT_DIR / config.stage / run_id
    run_dir.mkdir(parents=True, exist_ok=True)
    positive_rows, source_info = load_positive_source(config.clustering, seed=config.seed)
    train_positive, valid_positive, train_pairs, valid_pairs, warnings, size_info = split_and_make_pairs(
        positive_rows,
        config.negative_strategy,
        config.seed,
    )
    for warning in warnings:
        print(f'Warning: {warning}')

    device = get_device()
    train_loader, valid_loader = build_loaders(train_pairs, valid_pairs, tokenizer)
    model = BinaryESMClassifier(MODEL_NAME, config.freeze_esm, HIDDEN_DIM, DROPOUT).to(device)
    parameter_summary = count_parameters(model)
    lr = LR_FROZEN if config.freeze_esm else LR_UNFROZEN
    optimizer = AdamW((p for p in model.parameters() if p.requires_grad), lr=lr, weight_decay=WEIGHT_DECAY)
    loss_fn = nn.BCEWithLogitsLoss()
    history = []
    best_mcc = -1.0
    best_threshold = 0.5
    best_state = None
    checkpoint_path = run_dir / 'best_model.pt'

    print(f'Run: {config.stage} | {config.clustering} | {config.negative_strategy} | {architecture_name(config.freeze_esm)} | seed {config.seed}')
    for epoch in range(1, EPOCHS + 1):
        train_loss = train_one_epoch(model, train_loader, optimizer, device, loss_fn)
        labels, probs, losses = predict_model(model, valid_loader, device, loss_fn)
        valid_metrics = add_loss(select_validation_metrics(labels, probs), losses)
        valid_metrics['epoch'] = epoch
        valid_metrics['train_loss'] = train_loss
        history.append(valid_metrics)
        print(f'Epoch {epoch}: valid_mcc={valid_metrics["mcc"]:.4f}, threshold={valid_metrics["threshold"]:.2f}')
        if valid_metrics['mcc'] > best_mcc:
            best_mcc = valid_metrics['mcc']
            best_threshold = valid_metrics['threshold']
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            torch.save({
                'model_state_dict': model.state_dict(),
                'config': make_json_safe(config.__dict__),
                'threshold': best_threshold,
                'validation_metrics': valid_metrics,
                'parameter_summary': parameter_summary,
            }, checkpoint_path)

    if best_state is not None:
        model.load_state_dict(best_state)
        model.to(device)
    history_df = pd.DataFrame(history)
    history_df.to_csv(run_dir / 'training_history.csv', index=False)
    labels, probs, losses = predict_model(model, valid_loader, device, loss_fn)
    final_valid_metrics = add_loss(metrics_with_fixed_threshold(labels, probs, best_threshold), losses)
    valid_predictions = prediction_frame(valid_pairs, probs, best_threshold)
    valid_predictions.to_csv(run_dir / 'validation_predictions.csv', index=False)

    heldout_df = None
    if evaluate_heldout:
        heldout_rows = []
        for name, positive_frame in [
            ('test_1_sample', test_singleton_raw),
            ('test_2_to_10_samples', test_low_sample_raw),
        ]:
            heldout_pairs = make_heldout_pairs(positive_frame, config.negative_strategy, config.seed + 100)
            held_metrics, held_preds = evaluate_pairs(model, heldout_pairs, tokenizer, device, loss_fn, best_threshold)
            held_metrics['dataset'] = name
            held_metrics['rows'] = int(len(heldout_pairs))
            held_metrics['used_for_selection'] = False
            heldout_rows.append(held_metrics)
            held_preds.to_csv(run_dir / f'{name}_predictions.csv', index=False)
        heldout_df = pd.DataFrame(heldout_rows)
        heldout_df.to_csv(run_dir / 'heldout_metrics.csv', index=False)

    summary = {
        'run_id': run_id,
        'stage': config.stage,
        'clustering': config.clustering,
        'negative_strategy': config.negative_strategy,
        'freeze_esm': config.freeze_esm,
        'architecture': architecture_name(config.freeze_esm),
        'seed': config.seed,
        'selection_metric': 'validation_mcc',
        'validation_threshold': best_threshold,
        'validation_metrics': final_valid_metrics,
        'dataset_size': {**source_info, **size_info},
        'warnings': warnings,
        'parameter_summary': parameter_summary,
        'learning_rate': lr,
        'checkpoint_path': checkpoint_path,
        'run_dir': run_dir,
    }
    if heldout_df is not None:
        summary['heldout_metrics'] = heldout_df.to_dict(orient='records')
    save_run_summary(run_dir, summary, history_df, heldout_df)

    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return summary

## 9. Stage Selection Logic

Build the staged experiment grid and select the next stage from validation evidence only.

In [9]:
def config_key(summary: dict) -> tuple:
    return (summary['clustering'], summary['negative_strategy'], summary['freeze_esm'])

def index_row(summary: dict) -> dict:
    row = {
        'run_id': summary['run_id'],
        'stage': summary['stage'],
        'clustering': summary['clustering'],
        'negative_strategy': summary['negative_strategy'],
        'architecture': summary['architecture'],
        'freeze_esm': summary['freeze_esm'],
        'seed': summary['seed'],
        'validation_mcc': summary['validation_metrics']['mcc'],
        'validation_threshold': summary['validation_threshold'],
        'run_dir': str(summary['run_dir']),
    }
    for key, value in summary['dataset_size'].items():
        row[key] = value
    return row

def choose_best_by_mean_validation_mcc(summaries: list[dict]) -> tuple[tuple, pd.DataFrame]:
    rows = []
    for summary in summaries:
        rows.append({
            'config_key': config_key(summary),
            'clustering': summary['clustering'],
            'negative_strategy': summary['negative_strategy'],
            'freeze_esm': summary['freeze_esm'],
            'architecture': summary['architecture'],
            'seed': summary['seed'],
            'validation_mcc': summary['validation_metrics']['mcc'],
            'validation_threshold': summary['validation_threshold'],
        })
    df = pd.DataFrame(rows)
    grouped = df.groupby(['clustering', 'negative_strategy', 'freeze_esm', 'architecture']).agg(
        mean_validation_mcc=('validation_mcc', 'mean'),
        std_validation_mcc=('validation_mcc', 'std'),
        n_seeds=('seed', 'nunique'),
        per_seed_mcc=('validation_mcc', lambda values: list(values)),
        per_seed_threshold=('validation_threshold', lambda values: list(values)),
    ).reset_index()
    grouped['std_validation_mcc'] = grouped['std_validation_mcc'].fillna(0.0)
    grouped = grouped.sort_values('mean_validation_mcc', ascending=False).reset_index(drop=True)
    best = grouped.iloc[0]
    best_key = (best['clustering'], best['negative_strategy'], bool(best['freeze_esm']))
    return best_key, grouped

def run_configs(configs: list[ExperimentConfig], tokenizer, evaluate_heldout: bool = False) -> list[dict]:
    summaries = []
    for config in configs:
        summaries.append(train_experiment(config, tokenizer, evaluate_heldout=evaluate_heldout))
        pd.DataFrame([index_row(s) for s in all_run_summaries + summaries]).to_csv(OUTPUT_DIR / 'experiment_index.csv', index=False)
    return summaries

def require_8mer_if_needed(configs: list[ExperimentConfig]):
    if any(config.clustering == '8mer_clustering' for config in configs) and not TRAIN_8MER_CSV.exists():
        raise FileNotFoundError(
            f'Missing {TRAIN_8MER_CSV}. Run the 8-mer clustering notebook or copy the exported CSV into processed_data.'
        )

def stage1_configs() -> list[ExperimentConfig]:
    return [
        ExperimentConfig('stage1_clustering', clustering, 'scramble', True, seed)
        for clustering in ['no_clustering', 'p1_clustering', '8mer_clustering']
        for seed in SEEDS
    ]

def stage2_configs(best_clustering: str) -> list[ExperimentConfig]:
    return [
        ExperimentConfig('stage2_sampling', best_clustering, negative_strategy, True, seed)
        for negative_strategy in ['scramble', 'random_site', 'cross_family', 'mixed']
        for seed in SEEDS
    ]

def stage3_configs(best_clustering: str, best_negative_strategy: str) -> list[ExperimentConfig]:
    return [
        ExperimentConfig('stage3_architecture', best_clustering, best_negative_strategy, freeze_esm, seed)
        for freeze_esm in [True, False]
        for seed in SEEDS
    ]

## 10. Run Selected Sequential Stage

Execute the requested stage and save the experiment index for the completed runs.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
all_run_summaries = []
selection_rows = []

best_clustering = BEST_CLUSTERING_OVERRIDE
best_negative_strategy = BEST_NEGATIVE_STRATEGY_OVERRIDE
best_architecture_freeze = BEST_ARCHITECTURE_FREEZE_OVERRIDE

if RUN_STAGE in {'clustering', 'all'}:
    configs = stage1_configs()
    require_8mer_if_needed(configs)
    stage_summaries = run_configs(configs, tokenizer, evaluate_heldout=EVALUATE_HELDOUT_EVERY_RUN)
    all_run_summaries.extend(stage_summaries)
    best_key, stage_table = choose_best_by_mean_validation_mcc(stage_summaries)
    best_clustering = best_key[0]
    stage_table['stage'] = 'stage1_clustering'
    selection_rows.extend(stage_table.to_dict(orient='records'))
    print(f'Stage 1 best clustering: {best_clustering}')

if RUN_STAGE in {'sampling', 'all'}:
    if best_clustering is None:
        raise ValueError('Set best_clustering from Stage 1 before running Stage 2 alone.')
    configs = stage2_configs(best_clustering)
    stage_summaries = run_configs(configs, tokenizer, evaluate_heldout=EVALUATE_HELDOUT_EVERY_RUN)
    all_run_summaries.extend(stage_summaries)
    best_key, stage_table = choose_best_by_mean_validation_mcc(stage_summaries)
    best_negative_strategy = best_key[1]
    stage_table['stage'] = 'stage2_sampling'
    selection_rows.extend(stage_table.to_dict(orient='records'))
    print(f'Stage 2 best negative strategy: {best_negative_strategy}')

if RUN_STAGE in {'architecture', 'all'}:
    if best_clustering is None or best_negative_strategy is None:
        raise ValueError('Set best_clustering and best_negative_strategy before running Stage 3 alone.')
    configs = stage3_configs(best_clustering, best_negative_strategy)
    stage_summaries = run_configs(configs, tokenizer, evaluate_heldout=False)
    all_run_summaries.extend(stage_summaries)
    best_key, stage_table = choose_best_by_mean_validation_mcc(stage_summaries)
    best_architecture_freeze = best_key[2]
    stage_table['stage'] = 'stage3_architecture'
    selection_rows.extend(stage_table.to_dict(orient='records'))
    print(f'Stage 3 best architecture: {architecture_name(best_architecture_freeze)}')

if all_run_summaries:
    pd.DataFrame([index_row(summary) for summary in all_run_summaries]).to_csv(OUTPUT_DIR / 'experiment_index.csv', index=False)
if selection_rows:
    pd.DataFrame(selection_rows).to_csv(OUTPUT_DIR / 'stage_selection_summary.csv', index=False)

print('Stages complete.')

## 11. Final Held-Out Evaluation

Evaluate the validation-selected configuration on low-sample held-out proteases. This should not be used for model selection.

In [ ]:
def final_config_summaries() -> list[dict]:
    if best_clustering is None or best_negative_strategy is None or best_architecture_freeze is None:
        return []
    return [
        summary for summary in all_run_summaries
        if summary['stage'] == 'stage3_architecture'
        and summary['clustering'] == best_clustering
        and summary['negative_strategy'] == best_negative_strategy
        and summary['freeze_esm'] == best_architecture_freeze
    ]

def evaluate_heldout_from_checkpoint(summary: dict, tokenizer) -> list[dict]:
    device = get_device()
    checkpoint = torch.load(summary['checkpoint_path'], map_location=device)
    model = BinaryESMClassifier(MODEL_NAME, summary['freeze_esm'], HIDDEN_DIM, DROPOUT).to(device)
    model.load_state_dict(checkpoint['model_state_dict'])
    model.eval()
    loss_fn = nn.BCEWithLogitsLoss()
    threshold = summary['validation_threshold']
    rows = []
    run_dir = Path(summary['run_dir'])
    for name, positive_frame in [
        ('test_1_sample', test_singleton_raw),
        ('test_2_to_10_samples', test_low_sample_raw),
    ]:
        heldout_pairs = make_heldout_pairs(positive_frame, summary['negative_strategy'], summary['seed'] + 100)
        metrics, preds = evaluate_pairs(model, heldout_pairs, tokenizer, device, loss_fn, threshold)
        metrics['dataset'] = name
        metrics['rows'] = int(len(heldout_pairs))
        metrics['seed'] = summary['seed']
        metrics['used_for_selection'] = False
        rows.append(metrics)
        preds.to_csv(run_dir / f'{name}_predictions.csv', index=False)
    heldout_df = pd.DataFrame(rows)
    heldout_df.to_csv(run_dir / 'heldout_metrics.csv', index=False)
    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return rows

final_summaries = final_config_summaries()
if final_summaries:
    heldout_rows = []
    for summary in final_summaries:
        heldout_rows.extend(evaluate_heldout_from_checkpoint(summary, tokenizer))
    heldout_df = pd.DataFrame(heldout_rows)
    heldout_df.to_csv(OUTPUT_DIR / 'final_heldout_metrics.csv', index=False)
    per_seed_mcc = {str(summary['seed']): summary['validation_metrics']['mcc'] for summary in final_summaries}
    per_seed_threshold = {str(summary['seed']): summary['validation_threshold'] for summary in final_summaries}
    final_selected_config = {
        'clustering': best_clustering,
        'negative_strategy': best_negative_strategy,
        'freeze_esm': best_architecture_freeze,
        'architecture': architecture_name(best_architecture_freeze),
        'validation_threshold': float(np.mean(list(per_seed_threshold.values()))),
        'per_seed_validation_threshold': per_seed_threshold,
        'mean_validation_mcc': float(np.mean(list(per_seed_mcc.values()))),
        'per_seed_validation_mcc': per_seed_mcc,
        'heldout_metrics': heldout_df.to_dict(orient='records'),
        'heldout_used_for_selection': False,
        'selection_metric': 'validation_mcc',
    }
    (OUTPUT_DIR / 'final_selected_config.json').write_text(
        json.dumps(make_json_safe(final_selected_config), indent=2),
        encoding='utf-8',
    )
    print(f'Saved final_selected_config.json to {OUTPUT_DIR}')
else:
    print('No final config selected yet.')